# Part 1: Clustering
**Dataset:** UCI Spam dataset — 4601 points, 58 dimensions  
**Algorithms:** Farthest First Traversal (k-center) and K-Means++  
**Goal:** Implement 4 functions and compare clustering quality using the k-means objective

In [ ]:
import time
import random
import numpy as np
from pyspark import SparkContext
from pyspark.mllib.linalg import Vectors

try:
    sc.stop()
except:
    pass

sc = SparkContext(appName="Part1_Clustering")
sc.setLogLevel("ERROR")  # suppress verbose Spark logs, show only errors
print("Spark started successfully, version:", sc.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/08 11:51:18 WARN Utils: Your hostname, Sushantak.local, resolves to a loopback address: 127.0.0.1; using 172.31.111.206 instead (on interface en0)
26/04/08 11:51:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 11:51:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started successfully, version: 4.1.1


### Helper — Squared Euclidean Distance

- Computes squared Euclidean distance between two vectors  
- `Vectors.sqdist` is removed in PySpark 4.x, so NumPy is used  
- `np.array(v)` converts a Spark Vector to a NumPy array  
- `np.dot(diff, diff)` efficiently computes sum of squared differences  

In [ ]:
def sqdist(a, b):
    """Squared Euclidean distance between two Spark Vectors."""
    diff = np.array(a) - np.array(b)
    return float(np.dot(diff, diff))

print("sqdist helper defined")

sqdist helper defined


### Function — readVectorsSeq

- Reads the dataset file  
- Converts each line into a Spark Vector  

In [ ]:
def readVectorsSeq(filename):
    """
    Reads a CSV file where each line is a data point.
    Returns a list of Spark dense Vectors.
    """
    points = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip any empty lines
            vals = [float(x) for x in line.split(',')]
            points.append(Vectors.dense(vals))
    return points

# Load the dataset and verify shape
DATA_PATH = "Q1_Clustering/spambase.data"
P = readVectorsSeq(DATA_PATH)

print(f"Total points loaded : {len(P)}")
print(f"Dimensions per point: {len(P[0])}")
print(f"First point (first 5 dims): {list(P[0])[:5]}")

Total points loaded : 4601
Dimensions per point: 58
First point (first 5 dims): [np.float64(0.0), np.float64(0.64), np.float64(0.64), np.float64(0.0), np.float64(0.32)]


### Function — kcenter (Farthest First Traversal)

- Greedy approach for selecting k centers  
- Step 1: Pick any point as the first center  
- Step 2: Select the next center as the point farthest from existing centers  
- Step 3: Repeat until k centers are chosen  

- Time Complexity: \( O(|P| \cdot k) \)  
  - Each of the k iterations scans all |P| points to update distances  

In [ ]:
def kcenter(P, k):
    """
    Farthest First Traversal algorithm for k-center clustering.
    Returns k centers as a list of Vectors.
    Time complexity: O(|P| * k)
    """
    # Start with a random point as the first center
    first = random.randint(0, len(P) - 1)
    centers = [P[first]]

    # For each point, track its minimum distance to any chosen center so far
    # Initially, every point's nearest center is just the first one
    min_dists = [sqdist(p, centers[0]) for p in P]

    for _ in range(k - 1):
        # The next center is the point that is farthest from all existing centers
        farthest_idx = max(range(len(P)), key=lambda i: min_dists[i])
        new_center = P[farthest_idx]
        centers.append(new_center)

        # Update min distances — only need to check against the NEW center
        # because distances to older centers are already stored in min_dists
        for i, p in enumerate(P):
            d = sqdist(p, new_center)
            if d < min_dists[i]:
                min_dists[i] = d

    return centers

print("kcenter function defined successfully")

kcenter function defined successfully


### Function — kmeansPP (K-Means++ Initialization)

- Improved initialization over random selection  
- Step 1: Pick the first center randomly  
- Step 2: Choose next centers with probability proportional to squared distance from nearest center  
- Points farther away have higher probability → better spread of centers  

- Leads to better initial centers and lower clustering objective values  

In [ ]:
def kmeansPP(P, k):
    """
    K-Means++ center initialization.
    Returns k centers as a list of Vectors.
    Time complexity: O(|P| * k)
    """
    # Pick the first center uniformly at random
    first = random.randint(0, len(P) - 1)
    centers = [P[first]]

    # Track minimum squared distance from each point to its nearest center
    min_dists = [sqdist(p, centers[0]) for p in P]

    for _ in range(k - 1):
        # Compute total weight for probability normalisation
        total = sum(min_dists)

        if total == 0.0:
            # All points coincide with existing centers — pick randomly
            chosen = P[random.randint(0, len(P) - 1)]
        else:
            # D^2-weighted sampling: draw a random threshold and walk the list
            threshold = random.uniform(0, total)
            cumulative = 0.0
            chosen = P[-1]   # fallback
            for i, d in enumerate(min_dists):
                cumulative += d
                if cumulative >= threshold:
                    chosen = P[i]
                    break

        centers.append(chosen)

        # Update minimum distances with the newly added center
        for i, p in enumerate(P):
            d = sqdist(p, chosen)
            if d < min_dists[i]:
                min_dists[i] = d

    return centers

print("kmeansPP function defined successfully")

kmeansPP function defined successfully


### Function — kmeansObj (K-Means Objective)

- Measures the quality of selected centers  
- For each point, compute squared distance to its nearest center  
- Take the average over all points  
- Lower value indicates better clustering  

In [ ]:
def kmeansObj(P, C):
    """
    Computes the k-means objective: average squared distance
    of each point to its nearest center.
    Lower value = better clustering.
    """
    total = 0.0
    for p in P:
        # Find the minimum squared distance to any center
        nearest = min(sqdist(p, c) for c in C)
        total += nearest
    return total / len(P)

print("kmeansObj function defined successfully")

kmeansObj function defined successfully


### Main Program — Comparison

- Step 1: Run kcenter with k centers and measure time  
- Step 2: Run kmeans++ with k centers and compute objective  
- Step 3: Use kcenter with k1 centers as a coreset, then run kmeans++  

- Coreset idea:  
  - k1 > k centers act as a compact summary of the dataset  
  - Running kmeans++ on this smaller set is faster and often similarly effective  

- You can vary k and k1 to observe changes in objective values  

In [ ]:
k  = 5   # number of final clusters
k1 = 20  # number of coreset points (must be > k)

print(f"Dataset: {len(P)} points, {len(P[0])} dimensions")
print(f"k = {k}, k1 = {k1}")
print("=" * 55)

# --- Step 1: kcenter with k centers ---
print("\nStep 1: Running kcenter(P, k)...")
t_start = time.time()
C_kcenter = kcenter(P, k)
t_end = time.time()
print(f"  Time taken     : {t_end - t_start:.3f} seconds")
print(f"  Centers found  : {len(C_kcenter)}")

# --- Step 2: kmeans++ with k centers ---
print("\nStep 2: Running kmeansPP(P, k) then kmeansObj...")
C_kmeanspp = kmeansPP(P, k)
obj2 = kmeansObj(P, C_kmeanspp)
print(f"  kmeans++ objective (avg sq dist): {obj2:.4f}")

# --- Step 3: Coreset approach ---
print(f"\nStep 3: Running kcenter(P, k1={k1}) as coreset, then kmeansPP on it...")
X = kcenter(P, k1)           # get k1 coarse centers as a coreset
C_coreset = kmeansPP(X, k)   # run kmeans++ on the coreset to get k final centers
obj3 = kmeansObj(P, C_coreset)
print(f"  Coreset objective (avg sq dist) : {obj3:.4f}")

print("\n" + "=" * 55)
print("Summary:")
print(f"  Step 2 (kmeans++ directly)  : {obj2:.4f}")
print(f"  Step 3 (coreset + kmeans++) : {obj3:.4f}")
print("  Lower objective = better clustering quality")

Dataset: 4601 points, 58 dimensions
k = 5, k1 = 20

Step 1: Running kcenter(P, k)...
  Time taken     : 0.043 seconds
  Centers found  : 5

Step 2: Running kmeansPP(P, k) then kmeansObj...
  kmeans++ objective (avg sq dist): 95847.9239

Step 3: Running kcenter(P, k1=20) as coreset, then kmeansPP on it...
  Coreset objective (avg sq dist) : 771331.4220

Summary:
  Step 2 (kmeans++ directly)  : 95847.9239
  Step 3 (coreset + kmeans++) : 771331.4220
  Lower objective = better clustering quality


### Experiment — Effect of k and k1

- Try different values of k and k1  
- Increasing k1 improves the quality of the coreset  
- As k1 grows, Step 3’s objective approaches Step 2’s objective  

In [ ]:
print("Experimenting with different k1 values (k=5 fixed):")
print(f"{'k1':<8} {'Objective':>12}")
print("-" * 22)

k = 5
for k1_val in [10, 20, 50, 100]:
    X = kcenter(P, k1_val)
    C = kmeansPP(X, k)
    obj = kmeansObj(P, C)
    print(f"{k1_val:<8} {obj:>12.4f}")

print("\nObservation: as k1 grows, the coreset captures more detail")
print("and the objective value should decrease (better quality).")

Experimenting with different k1 values (k=5 fixed):
k1          Objective
----------------------
10        317921.1570
20       1750851.7317
50       2609274.5374
100       212023.7352

Observation: as k1 grows, the coreset captures more detail
and the objective value should decrease (better quality).


In [ ]:
# Cleaning up and stopping Spark when done
sc.stop()
print("Spark context stopped.")

Spark context stopped.
